In [12]:
!pip install datasets -q
from datasets import load_dataset
import pandas as pd


data = load_dataset("nyu-mll/glue","qqp")
df = data["train"].to_pandas()
df = df.rename(columns={"label":  "is_duplicate"})
df = df.sample(50000,random_state=42).reset_index(drop=True)


print(df.shape)
print(df.head)
print(df["is_duplicate"].value_counts())

(50000, 4)
<bound method NDFrame.head of                                                question1  \
0      Why do some people think that having a baby is...   
1      How vulnerable are paratroopers on their desce...   
2                   Examples of sole proprietorship? The   
3      Why don't I get answers for some of my questio...   
4                   Do Kashmiris wants to join Pakistan?   
...                                                  ...   
49995  How do you block someone by your smart phone s...   
49996                          What is meant by biology?   
49997  My university refuses to give me assignment ex...   
49998                  "What does it mean ""to learn""?"   
49999         Which are the books for self learning c++?   

                                               question2  is_duplicate     idx  
0                       Why is having a baby a blessing?             1  215130  
1                Are conventional paratroopers obsolete?             0  2346

In [13]:
import re

def clean_text(text):
 text = str(text).lower()
 text = text.replace("won't","will not")
 text = text.replace("can't","cannot")
 text = text.replace("n't","not")
 text = text.replace("'re","are")
 text = text.replace("'s","is")
 text = text.replace("'ve","have")
 text = text.replace("'ll","will")
 text = text.replace("'m","am")
 text = text.replace("?"," ")
 text = re.sub(r"[^a-z0-9]"," ",text)
 text = re.sub(r"\s+"," ",text).strip()
 return text
df["question1"]=df["question1"].apply(clean_text)
df["question2"]=df["question2"].apply(clean_text)

print(df[["question1","question2"]].head())

                                           question1  \
0  why do some people think that having a baby is...   
1  how vulnerable are paratroopers on their desce...   
2                examples of sole proprietorship the   
3  why donot i get answers for some of my questio...   
4                do kashmiris wants to join pakistan   

                                           question2  
0                    why is having a baby a blessing  
1             are conventional paratroopers obsolete  
2               how are sole proprietorships started  
3  why do some questions get more answers here in...  
4  do the people of kashmir want to join pakistan...  


In [14]:
!pip install rapidfuzz -q
from rapidfuzz import fuzz

df["q1_len"] = df["question1"].apply(len)
df["q2_len"] = df["question2"].apply(len)
df["q1_words"] = df["question1"].apply(lambda x: len(x.split()))
df["q2_words"] = df["question2"].apply(lambda x: len(x.split()))
df["common_words"] = df.apply(lambda r: len(set(r["question1"].split()) & set(r["question2"].split())), axis=1)
df["total_words"] = df["q1_words"] + df["q2_words"]
df["word_share"] = df["common_words"] / df["total_words"].replace(0, 1)
df["fuzz_ratio"] = df.apply(lambda r: fuzz.ratio(r["question1"], r["question2"]), axis=1)
df["fuzz_token_sort"] = df.apply(lambda r: fuzz.token_sort_ratio(r["question1"], r["question2"]), axis=1)
df["fuzz_token_set"] = df.apply(lambda r: fuzz.token_set_ratio(r["question1"], r["question2"]), axis=1)

print(df.columns.tolist())

['question1', 'question2', 'is_duplicate', 'idx', 'q1_len', 'q2_len', 'q1_words', 'q2_words', 'common_words', 'total_words', 'word_share', 'fuzz_ratio', 'fuzz_token_sort', 'fuzz_token_set']


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score
from xgboost import XGBClassifier
feature_cols =["q1_len" ,"q2_len","q1_words","q2_words","common_words","total_words","word_share","fuzz_ratio","fuzz_token_sort","fuzz_token_set"]
X = df[feature_cols]
y= df["is_duplicate"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.3,random_state=42,stratify=y
)
scaler = StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

lr=LogisticRegression(max_iter=1000)
lr.fit(X_train_s,y_train)
print("Logistic Regression log loss:",log_loss(y_test,lr.predict_proba(X_test_s)))
print("Logistic Regression accuracy:",accuracy_score(y_test,lr.predict(X_test_s)))

xgb=XGBClassifier(n_estimators=300, max_depth=5,learning_rate=0.1,eval_metric="logloss")
xgb.fit(X_train,y_train)
print("XGBoost log loss:",log_loss(y_test,xgb.predict_proba(X_test)))
print("XGBoost accuracy:",accuracy_score(y_test,xgb.predict(X_test)))

Logistic Regression log loss: 0.558876840862324
Logistic Regression accuracy: 0.6568
XGBoost log loss: 0.49862499487600315
XGBoost accuracy: 0.7155333333333334


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
tfidf.fit(pd.concat([df["question1"], df["question2"]]))

q1_vec = tfidf.transform(df["question1"])
q2_vec = tfidf.transform(df["question2"])

df["tfidf_cosine"] = 1 - paired_cosine_distances(q1_vec, q2_vec)

feature_cols = feature_cols + ["tfidf_cosine"]
X = df[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

xgb2 = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, eval_metric="logloss")
xgb2.fit(X_train, y_train)
print("XGBoost + TF-IDF log loss:", log_loss(y_test, xgb2.predict_proba(X_test)))
print("XGBoost + TF-IDF accuracy:", accuracy_score(y_test, xgb2.predict(X_test)))

XGBoost + TF-IDF log loss: 0.4886060972703602
XGBoost + TF-IDF accuracy: 0.7257333333333333
